# Essential advisor walkthrough
Only physical/effective-channel matrices, time/frequency bases and BER versus SNR. One selected matrix labeled Learned with OTFS/AFDM throughout. Selection is post hoc; see docs/SELECTION.md for limits.
No exploratory study generation, training or BER evaluation in this notebook.


In [ ]:
import os
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','VECLIB_MAXIMUM_THREADS']:
    os.environ[key] = '1'
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from advisor.core import configure
from advisor.artifacts import load_reference
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
artifacts = Path(os.environ.get('ADVISOR_ARTIFACTS', str(root/'local/artifacts')))
if not artifacts.exists():
    raise FileNotFoundError('Provision local bundle first; see README. No old source-project paths are used.')
configure()


## H → W → A
Columns of W synthesize symbols. Original ordering is authoritative; saved group permutation is an additional fixed view.


In [ ]:
data = np.load(artifacts/'focused/data/channels.npz', allow_pickle=False)
w = load_reference(artifacts)
h = torch.from_numpy(data['H'])
a = w.mH @ h @ w
print('H,W,A:', tuple(h.shape),tuple(w.shape),tuple(a.shape),a.dtype,a.device)
print('unitarity:',float(torch.linalg.matrix_norm(w.mH@w-torch.eye(64))))
print('paths:',data['tau'],data['nu'], 'normalization:',data['normalization'])


## Core figures
Default25dB view only; full numerical grids retained through30/40dB. Main raw slicer; fixed-Q and full receivers separated. Generate figures with the README command first.


In [ ]:
from IPython.display import display, Image
for name in ['channel_matrices_native','basis_time','basis_frequency','ber_QPSK_raw','ber_16QAM_raw']:
    path = Path(os.environ.get('ADVISOR_FIGURES', str(root/'local/figures')))/f'{name}.png'
    if path.exists(): display(Image(filename=str(path), width=1000))
    else: print('Generate first:',path.name)
summary = pd.read_csv(artifacts/'focused/ber_summary.csv')
display(summary[(summary.snr_db==25)&(summary.slicer=='raw')][['modulation','waveform','receiver','errors','bits','ber','ci_low','ci_high']])
